# 3.02 Chroma Vector Store

**Chroma**는 로컬에서 동작하는 오픈소스 벡터 DB. `persist_directory`에 파일 기반 저장소를 두고, 노트북 재시작 후에도 인덱스를 유지한다. HTTP 모드·Chroma Cloud 모드도 지원한다.

## 학습 목표

- `langchain-chroma` 패키지의 `Chroma` 클래스를 초기화한다
- `persist_directory`로 자동 영속화 (v0.4+부터 `.persist()` 호출 불필요)
- Chroma 메타데이터 필터 DSL (`$eq`, `$and`, `$or`, `$in`)
- MMR·점수 조회

## 언제 쓰나

- 단일 서버·노트북 범위의 로컬 RAG
- Pinecone 같은 관리형으로 옮기기 전의 개발용 미러
- docker·k8s 없이 바로 시작하고 싶은 소규모 내부 도구

## 3.02.1 환경 설정

`langchain-chroma`가 내부적으로 `chromadb`를 설치한다. 외부 서비스는 필요 없고 OpenAI 임베딩 키만 있으면 된다.

In [ ]:
# !pip install -U langchain-chroma langchain-openai

import os
from dotenv import load_dotenv
load_dotenv(override=True)
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 필요 (임베딩용)"

import chromadb
print("chromadb version:", chromadb.__version__)

## 3.02.2 기본 사용법

`Chroma.from_documents(documents, embedding, persist_directory=..., collection_name=...)` 로 초기화. `persist_directory`가 없으면 자동 생성된다.

In [ ]:
import tempfile
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

docs = [
    Document(page_content="Chroma는 Python/JS 기반 오픈소스 임베딩 DB이다.", metadata={"source": "chroma", "year": 2023}),
    Document(page_content="FAISS는 근사 최근접 이웃 검색용 라이브러리이다.", metadata={"source": "faiss", "year": 2017}),
    Document(page_content="Qdrant는 Rust로 작성된 벡터 검색 엔진이다.", metadata={"source": "qdrant", "year": 2021}),
    Document(page_content="Weaviate는 Go로 작성된 하이브리드 검색 엔진이다.", metadata={"source": "weaviate", "year": 2019}),
    Document(page_content="Milvus는 대규모 벡터 검색을 위한 분산 시스템이다.", metadata={"source": "milvus", "year": 2019}),
]

persist_dir = tempfile.mkdtemp(prefix="chroma_")
store = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="vectorstores",
    persist_directory=persist_dir,
)
print("persist_dir:", persist_dir)
print("collection count:", store._collection.count())

In [ ]:
for d in store.similarity_search("Rust 기반 벡터 엔진", k=3):
    print("-", d.metadata["source"], "|", d.page_content[:40])

## 3.02.3 메타데이터 필터링 — Chroma DSL

Chroma 필터는 dict 형태이며 MongoDB 풍 연산자를 쓴다.

- 단일 조건: `{"source": "qdrant"}` 또는 명시적 `{"source": {"$eq": "qdrant"}}`
- `$and` / `$or`: `{"$and": [{"year": {"$gte": 2019}}, {"source": {"$ne": "milvus"}}]}`
- `$in` / `$nin`: `{"source": {"$in": ["faiss", "qdrant"]}}`

In [ ]:
# 2019년 이후 & milvus 제외
hits = store.similarity_search(
    "분산 벡터 검색",
    k=5,
    filter={"$and": [
        {"year": {"$gte": 2019}},
        {"source": {"$ne": "milvus"}},
    ]},
)
for d in hits:
    print("-", d.metadata["source"], d.metadata["year"], "|", d.page_content[:40])

## 3.02.4 점수 포함 · MMR

`similarity_search_with_score` 는 **거리**(낮을수록 가까움)를 반환한다. 정규화된 `[0, 1]` 유사도를 원하면 `similarity_search_with_relevance_scores` 를 쓴다.

In [ ]:
print("--- with_score (거리) ---")
for doc, dist in store.similarity_search_with_score("오픈소스 벡터 DB", k=3):
    print(f"{dist:.4f}  {doc.metadata['source']}")

print("\n--- with_relevance_scores (0~1 유사도) ---")
for doc, rel in store.similarity_search_with_relevance_scores("오픈소스 벡터 DB", k=3):
    print(f"{rel:.4f}  {doc.metadata['source']}")

print("\n--- MMR ---")
for d in store.max_marginal_relevance_search("벡터 검색 엔진", k=3, fetch_k=5, lambda_mult=0.4):
    print("-", d.metadata["source"])

## 3.02.5 영속성 · 재로드

Chroma 0.4+에서는 `persist_directory`를 지정하면 **자동 커밋**된다. `.persist()` 호출은 더 이상 필요 없다. 재로드는 같은 `persist_directory` + `collection_name` + `embedding_function` 으로 `Chroma(...)` 객체를 다시 만들면 된다.

In [ ]:
# store 참조를 버리고 순수 디렉터리만으로 재개
del store

reopened = Chroma(
    collection_name="vectorstores",
    embedding_function=embeddings,
    persist_directory=persist_dir,
)
print("reopened count:", reopened._collection.count())
for d in reopened.similarity_search("Go 언어로 만든 검색 엔진", k=2):
    print("-", d.metadata["source"], "|", d.page_content[:40])

## 3.02.6 증분 업데이트

- `add_documents(docs, ids=...)` — 새 문서 추가 (ID 지정 시 upsert)
- `update_document(document_id, document)` — 단일 업데이트
- `delete(ids=[...])` — 삭제
- `get()` — 현재 컬렉션 덤프 (디버깅용)

In [ ]:
new_ids = reopened.add_documents([
    Document(page_content="Elasticsearch는 BM25와 dense vector를 혼합한 하이브리드 검색을 지원한다.", metadata={"source": "elasticsearch", "year": 2010}),
], ids=["es-1"])
print("added:", new_ids, "total:", reopened._collection.count())

reopened.delete(ids=["es-1"])
print("after delete:", reopened._collection.count())

# 디렉터리 정리
import shutil
shutil.rmtree(persist_dir)

## 체크리스트

- [ ] `collection_name`을 명시적으로 지정 — 기본값 `"langchain"` 은 여러 앱 공존 시 충돌 위험
- [ ] 메타데이터 필터는 **Chroma DSL dict** — callable 함수는 지원하지 않는다
- [ ] 다중 프로세스 쓰기는 지원하지 않음 — 프로덕션 멀티 워커는 Chroma HTTP 서버 또는 Pinecone/Qdrant로
- [ ] `persist_directory` 디스크 공간 모니터링 필요

## 다음

- `03_pgvector.ipynb` — 기존 Postgres에 pgvector 애드온
- `05_retrievers/` — MultiVector · ParentDocument retriever

## 참고

- Chroma docs: https://docs.trychroma.com/
- LangChain 통합: https://python.langchain.com/docs/integrations/vectorstores/chroma